In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch

import pickle
from pathlib import Path

import sys
base_path = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(base_path / '2_Propensities'))
sys.path.insert(0, str(base_path / '4_Baselines'))
sys.path.insert(0, str(base_path / '4_Baselines' / '4.2_OutcomeModel'))

import MF_class as MF
import OM_class as OM
import SASRec_class as sasrec

# 1 Choosing Dataset

In [2]:
datasets = ['ml-1m', 'steam', 'goodreads']
DATASET = datasets[2]

print(f"Using dataset: {DATASET}")

Using dataset: goodreads


# 2 Loading Dataset and Propensities Model

In [3]:
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'
path = base_artifacts / 'Datasets' / 'Processed' / DATASET
test = pd.read_csv(path / 'test.csv')
with open(path / 'item_dict.pkl', 'rb') as f:
    item_dict = pickle.load(f)
title2id = {v: k for k, v in item_dict.items()}

# Load chosen pairs and item dictionary
with open(base_artifacts / 'Chosen_Pairs' / f'{DATASET}_chosen_pairs.pkl', 'rb') as f:
    chosen_pairs = pickle.load(f)

chosen_pairs_ids = [
    (title2id[title_A], title2id[title_B])
    for title_A, title_B in chosen_pairs
]

In [4]:
model_path = base_artifacts / 'Propensity_Models'

with open(model_path / f'MF_params_{DATASET}.pkl', 'rb') as f:
    loaded_params = pickle.load(f)

MF_model = MF.MatrixFactorizationTorch(
    n_users=loaded_params['n_users'], 
    n_items=loaded_params['n_items'], 
    n_factors=loaded_params['n_factors']
)

model_name = f'MF_model_{DATASET}'
MF_model.load(path=model_path / (model_name + '.pt'))
MF_model.eval()

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            7801
Number of items:            6384
Number of factors:          50
Learning rate:              0.001
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           20
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-04-10 15:46:26


MatrixFactorizationTorch()

# 3 Evaluate SASRec

In [5]:
model_path = base_artifacts / 'SASRec_Models'
with open(model_path / f'sasrec_{DATASET}_init_dict.pkl', 'rb') as f:
    init_dict_loaded = pickle.load(f)
sasrec_model = sasrec.SASRecTorch(**init_dict_loaded)
sasrec_model.load(model_path / f'sasrec_{DATASET}.pt')

/home/gouni/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Model loaded from /home/gouni/CausalI2I_artifacts/SASRec_Models/sasrec_goodreads.pt.
num_items:     6384
max_seq_len:   50
device:        cuda
batch_size:    2048
lr:            0.001
weight_decay:  0.0
num_epochs:    20
saved_at:      2026-01-06 09:58:04
note:          None


In [6]:
def make_sequence(cause_id):
    PAD = sasrec_model.num_items
    L = sasrec_model.max_seq_len
    seq = torch.full((1, L), PAD, dtype=torch.long, device=sasrec_model.device)
    seq[0, -1] = cause_id
    return seq

In [7]:
sasrec_model.eval()

sasrec_scores = {}
candidates = torch.arange(0, sasrec_model.num_items, device=sasrec_model.device)
for pair in tqdm(chosen_pairs_ids):
    cause_id, effect_id = pair
    seq = make_sequence(cause_id)
    candidates_scores = sasrec_model.predict_scores(seq, candidates).detach().cpu().numpy()[0]
    sasrec_scores[pair] = candidates_scores[effect_id]

  0%|          | 0/10000 [00:00<?, ?it/s]

In [8]:
with open(base_artifacts / 'Datasets' / 'Evaluated' / 'SASRec' / f'{DATASET}_sasrec_scores.pkl', 'wb') as f:
    pickle.dump(sasrec_scores, f)

# 4 Outcome Model

In [9]:
user_embeddings = MF_model.P.detach().numpy()[:-1]
item_embeddings = MF_model.Q.detach().numpy()[:-1]
user_bias = MF_model.b_u.detach().numpy()[:-1]
item_bias = MF_model.b_i.detach().numpy()[:-1]

OM_model = OM.OutcomeModel(
    user_embeddings = user_embeddings,
    item_embeddings = item_embeddings,
    user_bias = user_bias,
    item_bias = item_bias,
)

OM_model.load(path = base_artifacts / 'Outcome_Models' / f'OM_{DATASET}.pt')

Loaded OutcomeModel summary:
Model:             OutcomeModel
Embedding dim:     50
Loss:              BCE
Learning rate:     0.0002
Weight decay:      0.0001
Batch size:        8192
Epochs:            25
Use AMP:           True
Timestamp:         2026-04-10 16:45:22
Note:              none


In [10]:
from joblib import Parallel, delayed

OM_model.eval()
test_users = np.asarray(test['user_id'].unique())

def score_pair(pair):
    cause_id, effect_id = pair
    with torch.inference_mode():
        mu1 = OM_model.predict_outcome(
            u_list=test_users, i=cause_id, j=effect_id, x=1
        ).cpu().numpy()
        mu0 = OM_model.predict_outcome(
            u_list=test_users, i=cause_id, j=effect_id, x=0
        ).cpu().numpy()
    return pair, {0: mu0, 1: mu1}

results = Parallel(n_jobs=-1, prefer="processes")(
    delayed(score_pair)(pair) for pair in tqdm(chosen_pairs_ids)
)

om_scores = dict(results)

  0%|          | 0/10000 [00:00<?, ?it/s]

In [11]:
with open(base_artifacts / 'Datasets' / 'Evaluated' / 'Outcome_Model' / f'{DATASET}_om_scores.pkl', 'wb') as f:
    pickle.dump(om_scores, f)